# IWI experiment · Five spatial folds of the DHS clusters

Splits the clusters of `train_clusters.csv` (written by `02_Check_Dataset`) into five folds, A to E, for cross-validation. No patch of one fold overlaps a patch of another fold, each fold holds a similar share of every country's clusters, and the IWI of every country is spread over the folds (`STRATIFY`): its poorest, middle and richest third each lie in at least three folds. The notebook does in one run what `ToBe_04_CSV_DBScan`, `ToBe_05/06_KMeansForLargeClusters` and `ToBe_07_LabelFolds` did by hand. It writes five files next to `train_clusters.csv`:

| file | content |
|---|---|
| `train_clusters_folds.csv` | the rows of `train_clusters.csv` with three new columns: `dbscan_group`, `group` (final spatial group, `class` in the ToBe CSVs) and `fold` (A to E). Train on this list. |
| `folds_removed.csv` | the clusters left out to separate the pieces of a split group, with the round in which they were removed |
| `folds_by_country.csv` | clusters per country and fold, removed clusters, smallest and largest fold share |
| `folds_by_stratum.csv` | clusters per IWI third of every country (a part) and fold, smallest and largest fold share |
| `folds_map.png` | the five folds on a map |

**Method**

1. **DBSCAN** (as `ToBe_04`): haversine distance, `eps` = 9.50 km, `min_samples` = 1. Patches are 6.72 km squares (224 px of 30 m), so two clusters farther apart than the patch diagonal, 6.72 km × √2 = 9.50 km, can never share pixels. DBSCAN puts clusters that are closer, directly or through a chain of clusters, into one group, and a group always stays in one fold. With `SEPARATION = 'box'`, two clusters are neighbours only when their patches overlap (|dx| and |dy| both below 6.72 km), the exact condition for sharing pixels: the groups are smaller and splits remove fewer clusters.
2. **KMeans for big groups** (as `ToBe_05` and `ToBe_06`). A group is split only when its country cannot get five balanced folds without that split. KMeans proposes the pieces. Then the cluster with the most neighbours closer than 9.50 km in other pieces is removed, again and again, until no two pieces are that close. Of twenty KMeans starts, the split that removes the fewest clusters is kept. This repeats in rounds until every country is balanced and no IWI part is crowded (see below).
3. **Fold labels** (as `ToBe_07`, extended to the IWI parts). Groups, largest first, go to the fold where their country and their IWI parts have the fewest clusters so far, relative to what one fold may hold. A tie goes to the fold with the fewest urban clusters (for a mostly urban group) or rural clusters (for a mostly rural group). This spreads the big cities over the folds. A removed cluster whose remaining neighbours all ended up in one fold then goes back into that fold.

Balanced means that every fold holds 15 to 25 % of each country's clusters (`FOLD_TOL` in Settings), give or take one cluster for small countries. `ToBe_07` reached that only for some countries. Rwanda, for example, got 8 clusters in fold B and 396 in fold D.

**IWI parts** (new, 2026-11-09). Balancing only the number of clusters of a country is not enough. In the earlier folds, nearly all clusters of Cairo, Egypt's richest and brightest area, ended up in one fold. That fold was then out of distribution: the models trained on the other folds had never seen such clusters. Now every country is cut into `N_STRATA` equal parts by IWI (its poorest, middle and richest third; fewer parts for small countries), and no fold may hold more than `STRATUM_MAX` = 35 % of a part. So every part lies in at least three folds, and when one of them is the test fold, at least one other is a training fold (only one fold validates). A city that holds most of a part, like Cairo, is split into pieces the same way as a big group of an unbalanced country (a split for a part may remove up to `STRATUM_REMOVE` of the group). `STRATIFY = None` gives exactly the folds of before.

Section 5 checks the result: no two clusters in different folds are closer than 9.50 km (no two patches overlap with `SEPARATION = 'box'`), every country is in all five folds, every IWI part is in at least three folds and in the training folds of every test fold, and the IWI distribution of every fold.

**Running.** Check `WORK_DIR` in Settings and run the cells in order. The notebook takes a few minutes on a CPU. With the same `train_clusters.csv` and `SEED` it gives the same folds. Run it again whenever `02` writes a new `train_clusters.csv`, and retrain afterwards, because the folds change with the input. When the folds change, the earlier `train_clusters_folds.csv` is kept as `train_clusters_folds_until_<date>.csv`. Move the run folders of 04 to 13 aside before training again: those notebooks skip every run whose `result.json` exists.

# Import libraries

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import BallTree
import matplotlib.pyplot as plt

try:                                    # interactive map in section 7 (folium comes with geemap)
    import folium
    from folium.plugins import MeasureControl
except ImportError:
    folium = None

# KMeans warns when a group has fewer distinct places than pieces (clusters with identical coordinates)
warnings.filterwarnings('ignore', message='Number of distinct clusters')

# Settings

In [ ]:
# ---- input / output ------------------------------------------------------------------
WORK_DIR    = '/path/to/data/'   # QC_DIR of 02
IN_CSV      = os.path.join(WORK_DIR, 'train_clusters.csv')        # written by 02_Check_Dataset
OUT_CSV     = os.path.join(WORK_DIR, 'train_clusters_folds.csv')  # IN_CSV + dbscan_group, group, fold
REMOVED_CSV = os.path.join(WORK_DIR, 'folds_removed.csv')
COUNTRY_CSV = os.path.join(WORK_DIR, 'folds_by_country.csv')
STRATUM_CSV = os.path.join(WORK_DIR, 'folds_by_stratum.csv')
MAP_PNG     = os.path.join(WORK_DIR, 'folds_map.png')

# ---- folds -------------------------------------------------------------------------------
FOLDS   = ['A', 'B', 'C', 'D', 'E']          # fold names, as in ToBe_07_LabelFolds
N_FOLDS = len(FOLDS)
SEED    = 0                                  # KMeans starts and tie-breaks: same input + same SEED = same folds

# ---- spatial separation ------------------------------------------------------------------
PATCH_KM   = 224 * 30 / 1000                 # patch side in 01: 224 pixels of about 30 m = 6.72 km
MIN_SEP_KM = PATCH_KM * np.sqrt(2)           # 9.50 km, the patch diagonal: clusters farther apart cannot overlap
EARTH_R_KM = 6371.0088                       # mean Earth radius (the default of the haversine package)
SEPARATION = 'circle'                        # 'circle': clusters of different folds more than MIN_SEP_KM apart,
                                             #   as ToBe_04; 'box': only their patches may not overlap, that is
                                             #   |dx| or |dy| >= BOX_KM (exact; removes fewer clusters in cities)
BOX_KM     = PATCH_KM + 0.05                 # 'box': the patch side and 50 m to spare

# ---- balance -------------------------------------------------------------------------------
FOLD_TOL = 0.25                              # every fold must hold (1 +- FOLD_TOL) / N_FOLDS of each country's
SHARE_LO = (1 - FOLD_TOL) / N_FOLDS          #   clusters: 0.15 ... 0.25 for 5 folds; 0.5 allows 0.10 ... 0.30
SHARE_HI = (1 + FOLD_TOL) / N_FOLDS

# ---- spread of the label over the folds (2026-11-09) ----------------------------------------------
STRATIFY       = 'iwi'     # column of IN_CSV whose distribution every fold should share; None = only the
                           #   numbers of clusters per country are balanced (the folds of before)
N_STRATA       = 3         # per country, the clusters are cut into this many equal parts by STRATIFY (thirds)
MIN_STRATUM    = 30        # every part needs at least this many clusters; smaller countries get fewer parts
STRATUM_MAX    = 0.35      # no fold may hold more than this share of a part: every part is in >= 3 folds
STRATUM_REMOVE = 0.35      # a split for a part may remove up to this share of the group (countries: below)

# ---- splitting big groups ------------------------------------------------------------------
MIN_SPLIT_SIZE   = 10      # groups with fewer clusters are never split
MAX_REMOVE_SHARE = 0.20    # a split may remove at most this share of the group's clusters, otherwise it is not done
MAX_K            = 10      # at most this many KMeans pieces in one split
KMEANS_TRIES     = 20      # KMeans starts per split; the one that removes the fewest clusters is kept
CUT_OFF_K        = [3, 4, 5, 6]   # also try each single cluster of KMeans with these k, cut off from the rest
MAX_ROUNDS       = 30      # rounds of "assign folds, split the groups of unbalanced countries"
RESTORE_REMOVED  = True    # give removed clusters back when all their neighbours ended up in one fold

# ---- maps -----------------------------------------------------------------------------------
MAP_COUNTRY = None         # country for the interactive map; None = the one with the most removed clusters

# 1. Clusters to split into folds

In [ ]:
df = pd.read_csv(IN_CSV)
# the older CSV version used these names (01 and 02 rename them the same way)
df = df.rename(columns={'ClusterID': 'cluster_id', 'year.of.interview': 'year',
                        'month.of.interview': 'month', 'iwi_avg': 'iwi'})
if 'country' not in df:
    df['country'] = np.nan
# a missing country is taken from the cluster id: KE.Nairobi.8B.1643 -> KE
df['country'] = df['country'].fillna(df['cluster_id'].astype(str).str.split('.').str[0])

# 02 has already dropped these; kept here so the notebook also works on the full DHS table
no_xy = df[['lon', 'lat']].isna().any(axis=1) | ((df['lon'] == 0) & (df['lat'] == 0))
df = df[~no_xy]
dup = df['cluster_id'].duplicated(keep='first')
df = df[~dup].reset_index(drop=True)
print('clusters:', len(df), '| dropped, no coordinates:', int(no_xy.sum()), '| dropped, duplicate id:', int(dup.sum()))

N = len(df)
lat = df['lat'].to_numpy(float)
lon = df['lon'].to_numpy(float)
country_code, COUNTRIES = pd.factorize(df['country'], sort=True)   # country of every cluster as 0, 1, 2, ...
# urban clusters (rural == False); only used to break ties when the folds are labelled
URBAN = (df['rural'].astype(str).str.strip().str.lower().isin(['false', '0', 'u', 'urban']).to_numpy()
         if 'rural' in df else np.zeros(N, bool))
print(len(COUNTRIES), 'countries,', int(URBAN.sum()), 'urban clusters\n')
print(df['country'].value_counts().to_string())

# IWI parts: every country cut into N_STRATA equal parts by STRATIFY (fewer parts for small countries)
stratum = np.full(N, -1)                  # part of every cluster, -1 = in no part (no value or a small country)
STRATA, STRATUM_COUNTRY = [], []          # name ("Egypt iwi 75-100") and country code of every part
if STRATIFY:
    assert STRATIFY in df, f'{STRATIFY} is not a column of {IN_CSV}'
    value = pd.to_numeric(df[STRATIFY], errors='coerce')
    for c, name in enumerate(COUNTRIES):
        rows = np.flatnonzero((country_code == c) & value.notna().to_numpy())
        k = int(min(N_STRATA, len(rows) // MIN_STRATUM))
        if k < 2:
            continue
        part = pd.qcut(value.iloc[rows].rank(method='first'), k, labels=False).to_numpy()   # equal parts
        for p in range(k):
            v = value.iloc[rows[part == p]]
            STRATA.append(f'{name} {STRATIFY} {v.min():.0f}-{v.max():.0f}')
            STRATUM_COUNTRY.append(c)
            stratum[rows[part == p]] = len(STRATA) - 1
    print(f'\n{len(STRATA)} {STRATIFY} parts in {len(set(STRATUM_COUNTRY))} countries; '
          f'{int((stratum < 0).sum())} clusters in no part')

# 2. DBSCAN: groups of clusters closer than 9.50 km

The same DBSCAN as `ToBe_04` (`eps` = 9.50 km, `min_samples` = 1), but with scikit-learn's built-in haversine metric on a ball tree instead of a custom distance function. It takes seconds and does not build the full distance matrix. The pairs of clusters closer than 9.50 km are also kept as a sparse graph `A`, which sections 3 and 4 use. With `SEPARATION = 'box'`, two clusters are neighbours only when their patches overlap (|dx| and |dy| both below 6.72 km; such pairs are at most 9.50 km apart), and the groups are the connected parts of that graph. That is the exact condition for sharing pixels, so the groups are smaller and splits remove fewer clusters.

In [ ]:
t0 = time.time()
X = np.radians(np.c_[lat, lon])               # the haversine metric takes (lat, lon) in radians
EPS = (MIN_SEP_KM if SEPARATION == 'circle' else BOX_KM * np.sqrt(2)) / EARTH_R_KM   # as an angle
assert SEPARATION in ('circle', 'box'), SEPARATION


def neighbour_graph(X, eps):
    """Sparse symmetric matrix with a 1 for every pair of clusters at most eps apart (X in radians)."""
    ind = BallTree(X, metric='haversine').query_radius(X, r=eps)
    rows = np.repeat(np.arange(len(X)), [len(i) for i in ind])
    cols = np.concatenate(ind)
    off = rows != cols
    G = sp.csr_matrix((np.ones(off.sum(), np.int32), (rows[off], cols[off])), shape=(len(X), len(X)))
    return ((G + G.T) > 0).astype(np.int32).tocsr()


def patches_overlap(i, j):
    """True for the pairs of clusters (i, j) whose patches overlap: |dx| and |dy| both below BOX_KM."""
    phi = np.radians((lat[i] + lat[j]) / 2)
    dx = EARTH_R_KM * np.radians(lon[j] - lon[i]) * np.cos(phi)
    dy = EARTH_R_KM * np.radians(lat[j] - lat[i])
    return (np.abs(dx) < BOX_KM) & (np.abs(dy) < BOX_KM)


A = neighbour_graph(X, EPS)
if SEPARATION == 'box':                   # keep the pairs whose patches overlap; groups = connected parts
    coo = A.tocoo()
    m = patches_overlap(coo.row, coo.col)
    A = sp.csr_matrix((coo.data[m], (coo.row[m], coo.col[m])), shape=A.shape)
    dbscan_group = connected_components(A, directed=False)[1]
else:
    dbscan_group = DBSCAN(eps=EPS, min_samples=1, metric='haversine', algorithm='ball_tree').fit_predict(X)
df['dbscan_group'] = dbscan_group
n_cc, cc = connected_components(A, directed=False)
# DBSCAN with min_samples = 1 groups exactly the connected parts of this graph
assert n_cc == dbscan_group.max() + 1 == len(pd.DataFrame({'a': cc, 'b': dbscan_group}).drop_duplicates())
pairs = f'closer than {MIN_SEP_KM:.2f} km' if SEPARATION == 'circle' else 'with overlapping patches'
print(f'{n_cc} DBSCAN groups, {A.nnz // 2} pairs of clusters {pairs} ({time.time() - t0:.1f} s)\n')

sizes = df['dbscan_group'].value_counts()
bins = pd.cut(sizes, [0, 1, 5, 20, 100, 500, np.inf], labels=['1', '2-5', '6-20', '21-100', '101-500', '> 500'])
print('groups by number of clusters\n', bins.value_counts(sort=False).to_string(), '\n')

top = sizes.head(15)
countries_of = (df[df['dbscan_group'].isin(top.index)].groupby('dbscan_group')['country']
                .agg(lambda s: ', '.join(f'{c} {n}' for c, n in s.value_counts().items())))
print('largest groups\n', pd.DataFrame({'clusters': top, 'countries': countries_of[top.index]}).to_string())

In [ ]:
def country_groups(group, keep):
    """Per country: clusters, groups, and the share of the country's clusters in its largest group."""
    t = pd.DataFrame({'country': df['country'].to_numpy()[keep], 'group': group[keep]})
    per = t.groupby('country').agg(clusters=('group', 'size'), groups=('group', 'nunique'))
    per['largest_group_%'] = (100 * t.value_counts().groupby('country').max() / per['clusters']).round(1)
    return per.sort_values('largest_group_%', ascending=False)


before = country_groups(dbscan_group, np.ones(N, bool))
print(f'A country whose largest group holds more than {100 * SHARE_HI:.0f} % of its clusters cannot be balanced '
      'without splitting that group.\n')
print(before.to_string())

# 3. Break big groups with KMeans

Each round labels the folds with the method of section 4 and looks at the fold shares of every country and of every IWI part. For every country with a fold outside 15 to 25 %, and for every part with a fold above `STRATUM_MAX` (a crowded part), its largest group is split:

* **Limits.** No piece may hold more of a country's clusters than one fold may hold (25 %). If the group is already smaller than that, the limit for the unbalanced country is one fair share (20 %), and below that, half the group. In the same way no piece may hold more of an IWI part than one fold may hold (35 %), and for a crowded part whose largest group is already smaller, the limit is half of that group.
* **KMeans** on the clusters' coordinates in km proposes the pieces. Two kinds of candidates are tried. The first is the k pieces of KMeans, where k is the number of pieces the limits need (and k + 1). The second is each single cluster of KMeans with k = 3 to 6, cut off from the rest of the group. Cutting one piece off the edge of a dense group is often much cheaper than cutting through its middle.
* **Removing edge clusters** (`remove_edge_points` of `ToBe_05`). A cluster with no neighbour in its own piece, whose neighbours all lie in one other piece, first moves to that piece at no cost. Then the cluster with the most neighbours closer than 9.50 km in other pieces is removed, again and again, until no piece has a neighbour in another piece. The connected parts that remain are the new groups.
* **Best candidate.** Each of the 20 KMeans starts gives several candidates. The one that keeps all pieces within the limits and removes the fewest clusters wins. If none keeps the limits, the one closest to them wins, and the next round goes on from there. A split that would remove more than 20 % of the group (35 % for a crowded IWI part, `STRATUM_REMOVE`) is not done, and the group stays as it is.

The rounds stop when every country is balanced and no part is crowded, when nothing that is left can be split, or after `MAX_ROUNDS`.

In [ ]:
def assign_folds(group, country, urban, strat=None, seed=SEED):
    """Fold number (0 .. N_FOLDS-1) of every cluster: ToBe_07's rule, for all countries at once, extended to the
    IWI parts. A group touches cells: its countries and its parts. The load of a cell in a fold is the cell's
    clusters there relative to what one fold may hold (SHARE_HI of a country, STRATUM_MAX of a part).
    Groups go in decreasing order of their largest share (ties at random). Each group goes to the fold where the
    largest resulting load of its cells is smallest. Ties go to the fold with the fewest clusters of the group's
    kind (urban or rural, whichever the group mostly has), then to the fold with the fewest clusters, then at
    random. Without parts this is exactly the rule of the earlier version."""
    rng = np.random.default_rng(seed)
    n_c, n_s = len(COUNTRIES), len(STRATA)
    has = np.zeros(len(group), bool) if strat is None else strat >= 0
    cell = np.r_[country, n_c + strat[has]] if has.any() else country
    cell_group = np.r_[group, group[has]] if has.any() else group
    room = np.bincount(cell, minlength=n_c + n_s) * np.r_[np.full(n_c, SHARE_HI), np.full(n_s, STRATUM_MAX)]
    gc = pd.DataFrame({'g': cell_group, 'c': cell}).value_counts().reset_index(name='n').sort_values(['g', 'c'])
    g, c, n = gc['g'].to_numpy(), gc['c'].to_numpy(), gc['n'].to_numpy()
    start = np.flatnonzero(np.r_[True, g[1:] != g[:-1]])            # first row of every group (sorted by g)
    stop = np.r_[start[1:], len(g)]
    share = np.maximum.reduceat(n / room[c], start)                 # largest load of every group
    size = np.unique(group, return_counts=True)[1]                  # clusters per group, sorted by group id
    n_urban = pd.Series(urban).groupby(group).sum().to_numpy()      # also sorted by group id
    order = np.lexsort((rng.random(len(start)), -share))
    load = np.zeros((n_c + n_s, N_FOLDS))                           # clusters per cell and fold so far
    total = np.zeros(N_FOLDS)
    total_urban = np.zeros(N_FOLDS)
    fold_of_group = {}
    for j in order:
        cs, ns = c[start[j]:stop[j]], n[start[j]:stop[j]]
        cost = ((load[cs] + ns[:, None]) / room[cs, None]).max(axis=0)
        same_kind = total_urban if 2 * n_urban[j] >= size[j] else total - total_urban
        f = np.lexsort((rng.random(N_FOLDS), total, same_kind, cost))[0]
        load[cs, f] += ns
        total[f] += size[j]
        total_urban[f] += n_urban[j]
        fold_of_group[g[start[j]]] = f
    return pd.Series(group).map(fold_of_group).to_numpy()


def fold_table(fold, country):
    """Clusters per country (rows) and fold (columns)."""
    t = np.zeros((len(COUNTRIES), N_FOLDS), int)
    np.add.at(t, (country, fold), 1)
    return pd.DataFrame(t, index=COUNTRIES, columns=FOLDS)


def stratum_table(fold, strat):
    """Clusters per IWI part (rows) and fold (columns)."""
    t = np.zeros((len(STRATA), N_FOLDS), int)
    m = strat >= 0
    np.add.at(t, (strat[m], fold[m]), 1)
    return pd.DataFrame(t, index=STRATA, columns=FOLDS)


def outside_limits(counts):
    """How many clusters of one country (counts per fold) lie outside SHARE_LO ... SHARE_HI of the country,
    rounded outwards to whole clusters, so that 9 clusters as 1 2 2 2 2 still count as balanced."""
    counts = np.asarray(counts, float)
    lo, hi = np.floor(SHARE_LO * counts.sum()), np.ceil(SHARE_HI * counts.sum())
    return np.clip(lo - counts, 0, None).sum() + np.clip(counts - hi, 0, None).sum()


def stratum_over(counts):
    """How many clusters of one IWI part (counts per fold) lie above STRATUM_MAX of the part, rounded up."""
    counts = np.asarray(counts, float)
    return np.clip(counts - np.ceil(STRATUM_MAX * counts.sum()), 0, None).sum()


def unbalanced(table):
    """Countries with a fold outside their limits."""
    return [c for c, row in table.iterrows() if outside_limits(row) > 0]


def crowded(stable):
    """IWI parts (positions in STRATA) with a fold above STRATUM_MAX."""
    return [s for s, row in enumerate(stable.to_numpy()) if stratum_over(row) > 0]

In [ ]:
def local_km(lat, lon):
    """x, y in km around the centre of a group, so that KMeans measures distances in km."""
    x = EARTH_R_KM * np.radians(lon - lon.mean()) * np.cos(np.radians(lat.mean()))
    y = EARTH_R_KM * np.radians(lat - lat.mean())
    return np.c_[x, y]


def move_lonely(G, piece):
    """A cluster with no neighbour in its own piece, whose neighbours all lie in one other piece, joins that
    piece. Afterwards it touches no other piece, so it does not have to be removed."""
    piece = piece.copy()
    deg = np.diff(G.indptr)
    has = deg > 0
    if not has.any():
        return piece
    rows = np.repeat(np.arange(len(piece)), deg)
    for _ in range(100):
        nb_piece = piece[G.indices]                                   # piece of every neighbour
        same = np.bincount(rows, weights=(nb_piece == piece[rows]).astype(float), minlength=len(piece))
        lo = np.full(len(piece), -1)
        hi = np.full(len(piece), -2)
        lo[has] = np.minimum.reduceat(nb_piece, G.indptr[:-1][has])
        hi[has] = np.maximum.reduceat(nb_piece, G.indptr[:-1][has])
        moved = 0
        for i in np.flatnonzero(has & (same == 0) & (lo == hi)):
            nb = piece[G.indices[G.indptr[i]:G.indptr[i + 1]]]         # check again: earlier moves count
            if nb[0] != piece[i] and (nb == nb[0]).all():
                piece[i] = nb[0]
                moved += 1
        if not moved:
            break
    return piece


def remove_edge_clusters(G, piece):
    """ToBe_05's remove_edge_points: remove the cluster with the most neighbours in other pieces, again and
    again, until no two pieces touch. Returns a mask of the removed clusters."""
    coo = G.tocoo()
    other = piece[coo.row] != piece[coo.col]
    C = sp.csr_matrix((np.ones(other.sum(), np.int32), (coo.row[other], coo.col[other])), shape=G.shape)
    deg = np.diff(C.indptr).astype(np.int64)                          # neighbours in other pieces
    removed = np.zeros(len(piece), bool)
    while True:
        i = int(deg.argmax())
        if deg[i] == 0:
            return removed
        removed[i] = True
        deg[i] = 0
        nb = C.indices[C.indptr[i]:C.indptr[i + 1]]
        deg[nb[~removed[nb]]] -= 1


def kmeans_pieces(xy, k_way, seed=SEED):
    """The KMeans labellings to try: the k pieces of KMeans for every k in k_way, and each single KMeans
    cluster (k in CUT_OFF_K) cut off from the rest of the group. Cutting one piece off the edge of a dense
    group is often much cheaper than cutting it through the middle."""
    cut_off = [k for k in CUT_OFF_K if k < len(xy)]
    for t in range(KMEANS_TRIES):
        lab = {k: KMeans(n_clusters=k, n_init=1, random_state=seed + t).fit_predict(xy) for k in {*k_way, *cut_off}}
        for k in k_way:
            yield f'{k} pieces', t, lab[k]
        for k in cut_off:
            for piece in range(k):
                yield f'1 of {k} cut off', t, (lab[k] == piece).astype(int)


def split_group(members, limit):
    """Best KMeans split of the clusters `members` (one group) into pieces more than MIN_SEP_KM apart.
    limit = {cell: the most clusters of that cell one piece may hold}, where a cell is a country code, or
    len(COUNTRIES) + the position of an IWI part in STRATA. The split that keeps every piece within the limits and
    removes the fewest clusters wins; if no split keeps the limits, the one that comes closest to them."""
    G = A[members][:, members]
    xy = local_km(lat[members], lon[members])
    n_c = len(COUNTRIES)
    cs = np.array(list(limit))
    lim = np.array([limit[c] for c in cs], float)
    inside = np.stack([(country_code[members] == c) if c < n_c else (stratum[members] == c - n_c) for c in cs])
    is_country = cs < n_c
    need = max(np.ceil(inside[is_country].sum(axis=1) / lim[is_country]).sum(),   # pieces needed for the limits
               np.ceil(inside[~is_country].sum(axis=1) / lim[~is_country]).max() if (~is_country).any() else 0)
    k0 = int(np.clip(need, 2, min(MAX_K, len(members))))
    k_way = sorted({k0, min(k0 + 1, MAX_K, len(members))})
    best = None
    for how, t, piece in kmeans_pieces(xy, k_way):
        removed = remove_edge_clusters(G, move_lonely(G, piece))
        keep = np.flatnonzero(~removed)
        n_pieces, part = connected_components(G[keep][:, keep], directed=False)
        largest = np.array([np.bincount(part[inside[r, keep]], minlength=n_pieces).max() for r in range(len(cs))])
        over = (largest / lim).max()                                   # <= 1: all pieces within the limits
        score = (max(over, 1.0), int(removed.sum()))
        if best is None or score < best['score']:
            best = dict(score=score, over=over, removed=removed, keep=keep, part=part, n_pieces=n_pieces,
                        how=how, start=t)
    return best

In [ ]:
t0 = time.time()
group = dbscan_group.copy()          # current group of every cluster
kept = np.ones(N, bool)              # False = removed between two pieces
removed_round = np.full(N, -1)       # round in which a cluster was removed
frozen = set()                       # groups whose split was not done (too costly or no real split)
split_log = []
next_id = group.max() + 1
n_c = len(COUNTRIES)

for rnd in range(MAX_ROUNDS + 1):
    idx = np.flatnonzero(kept)
    fold = assign_folds(group[idx], country_code[idx], URBAN[idx], stratum[idx])
    table = fold_table(fold, country_code[idx])
    stable = stratum_table(fold, stratum[idx])
    bad, bad_s = unbalanced(table), crowded(stable)
    parts = f', {len(bad_s)} crowded {STRATIFY} parts {[STRATA[s] for s in bad_s]}' if STRATA else ''
    print(f'round {rnd}: {len(np.unique(group[idx]))} groups, {N - len(idx)} clusters removed, '
          f'{len(bad)} unbalanced countries {bad}{parts}')
    if not (bad or bad_s) or rnd == MAX_ROUNDS:
        break

    counts = pd.DataFrame({'group': group[idx], 'country': country_code[idx]}).value_counts()
    in_part = stratum[idx] >= 0
    s_counts = pd.DataFrame({'group': group[idx][in_part], 'stratum': stratum[idx][in_part]}).value_counts()
    s_groups = set(s_counts.index.get_level_values('group')) if len(s_counts) else set()
    n_country = table.sum(axis=1).to_numpy()
    n_stratum = stable.sum(axis=1).to_numpy()
    size_of = pd.Series(group[idx]).value_counts()
    targets = {}                                         # group -> the cells (countries, parts) it is largest in
    for c in COUNTRIES.get_indexer(bad):
        g = counts.xs(c, level='country').idxmax()
        if g not in frozen and size_of[g] >= MIN_SPLIT_SIZE:
            targets.setdefault(g, []).append(c)
    for s in bad_s:
        g = s_counts.xs(s, level='stratum').idxmax()
        if g not in frozen and size_of[g] >= MIN_SPLIT_SIZE:
            targets.setdefault(g, []).append(n_c + s)
    if not targets:
        print('nothing left to split: the largest group of every unbalanced country or crowded part is too small '
              'or too costly')
        break

    for g, why in sorted(targets.items()):
        members = idx[group[idx] == g]
        per_country = counts.xs(g, level='group')        # clusters of each country in the group
        per_stratum = s_counts.xs(g, level='group') if g in s_groups else pd.Series(dtype=int)
        limit, in_group = {}, {}
        for c, n in per_country.items():
            hi = np.ceil(SHARE_HI * n_country[c])         # the most clusters of c that one fold may hold
            fair = n_country[c] / N_FOLDS
            if n > hi:
                limit[c] = hi
            elif c in why:                                # the group fits into a fold but still blocks the
                limit[c] = max(np.floor(fair) if n > fair else np.ceil(n / 2), 1)   # balance: aim lower
            in_group[c] = n
        for s, n in per_stratum.items():
            hi = np.ceil(STRATUM_MAX * n_stratum[s])      # the most clusters of part s that one fold may hold
            if n > hi:
                limit[n_c + s] = hi
            elif n_c + s in why:                          # fits into a fold, but the part is still crowded
                limit[n_c + s] = max(np.ceil(n / 2), 1)
            in_group[n_c + s] = n
        split = split_group(members, limit)
        over_before = max(in_group[k] / limit[k] for k in limit)
        n_removed = int(split['removed'].sum())
        max_share = STRATUM_REMOVE if any(k >= n_c for k in why) else MAX_REMOVE_SHARE
        done = split['n_pieces'] >= 2 and split['over'] < over_before and n_removed <= max_share * len(members)
        split_log.append({'round': rnd, 'group': g, 'clusters': len(members),
                          'countries': ', '.join(f'{COUNTRIES[c]} {n}' for c, n in
                                                 per_country.sort_values(ascending=False).items()),
                          'for': ', '.join(COUNTRIES[k] if k < n_c else STRATA[k - n_c] for k in why),
                          'limit': ', '.join(f'{COUNTRIES[k] if k < n_c else STRATA[k - n_c]} {v:.0f}'
                                             for k, v in limit.items()),
                          'kmeans': split['how'], 'removed': n_removed,
                          'removed_%': round(100 * n_removed / len(members), 1),
                          'pieces': ' '.join(map(str, sorted(np.bincount(split['part']), reverse=True)[:12])),
                          'done': done})
        if not done:
            frozen.add(g)
            continue
        kept[members[split['removed']]] = False
        removed_round[members[split['removed']]] = rnd
        group[members[split['keep']]] = next_id + split['part']
        next_id += split['n_pieces']

split_log = pd.DataFrame(split_log)
print(f'\n{len(split_log)} splits in {time.time() - t0:.0f} s, {int((~kept).sum())} clusters removed '
      f'({100 * (~kept).mean():.2f} %)')
split_log

# 4. Label the folds

The folds of the last round, with `ToBe_07`'s rule as described in section 3, are the final ones. Then the removed clusters are checked one by one. A removed cluster goes back into the data when all its neighbours closer than 9.50 km are in one fold. It also goes back when it has no neighbour left, and then it joins its country's smallest fold. It is only given back if its country stays balanced and its IWI part does not become crowded. Finally, `group` is set to the connected parts of the 9.50 km graph among the clusters that are kept, and every group lies in one fold.

In [ ]:
fold = np.full(N, -1)
idx = np.flatnonzero(kept)
fold[idx] = assign_folds(group[idx], country_code[idx], URBAN[idx], stratum[idx])


def restore_removed(fold, kept):
    """Give back removed clusters that separate nothing: all their kept neighbours are in one fold."""
    fold, kept = fold.copy(), kept.copy()
    load = fold_table(fold[kept], country_code[kept]).to_numpy(copy=True)   # writable (pandas 3 returns read-only views)
    s_load = stratum_table(fold[kept], stratum[kept]).to_numpy(copy=True)
    restored, changed = 0, True
    while changed:
        changed = False
        for i in np.flatnonzero(~kept):
            nb = A.indices[A.indptr[i]:A.indptr[i + 1]]
            f = np.unique(fold[nb[kept[nb]]])
            if len(f) > 1:
                continue
            c = country_code[i]
            f = f[0] if len(f) else load[c].argmin()
            new = load[c].copy()
            new[f] += 1
            if outside_limits(new) > outside_limits(load[c]):
                continue                                   # would make the country's balance worse
            s = stratum[i]
            if s >= 0:
                new_s = s_load[s].copy()
                new_s[f] += 1
                if stratum_over(new_s) > stratum_over(s_load[s]):
                    continue                               # would crowd the cluster's IWI part
                s_load[s] = new_s
            kept[i], fold[i], load[c] = True, f, new
            restored += 1
            changed = True
    return fold, kept, restored


n_removed_split = int((~kept).sum())
if RESTORE_REMOVED and n_removed_split:
    fold, kept, n_restored = restore_removed(fold, kept)
    removed_round[kept] = -1
    print(f'{n_restored} of {n_removed_split} removed clusters given back, {int((~kept).sum())} stay removed')

# final groups: the connected parts of the 9.50 km graph among the kept clusters
idx = np.flatnonzero(kept)
_, final_group = connected_components(A[idx][:, idx], directed=False)
folds = df.loc[kept].copy()
folds['group'] = final_group
folds['fold'] = np.array(FOLDS)[fold[idx]]
removed = df.loc[~kept].copy()
removed['removed_in_round'] = removed_round[~kept]
assert (folds.groupby('group')['fold'].nunique() == 1).all()       # no group is in two folds

print(f"\n{len(folds)} clusters in {folds['group'].nunique()} groups, {len(removed)} removed\n")
print(folds['fold'].value_counts().sort_index().to_string())

# 5. Checks

1. The closest two clusters in different folds, measured again from the coordinates without the graph above. They must be more than 9.50 km apart.
2. For every country: clusters per fold, removed clusters, and the smallest and largest fold share.
3. For every IWI part: clusters per fold and the largest fold share, which may not exceed `STRATUM_MAX`.
4. The IWI label (mean, standard deviation and percentiles), urban and rural clusters, and the survey years per fold, which should be similar in all folds.
5. The IWI distribution of every fold, for all clusters and for the countries whose folds differ most.

In [ ]:
# 1. no two clusters of different folds closer than MIN_SEP_KM (patches cannot overlap)
Xk, fk = X[kept], fold[kept]
closest = np.inf
for f in range(N_FOLDS):
    d, _ = BallTree(Xk[fk != f], metric='haversine').query(Xk[fk == f], k=1)
    closest = min(closest, d.min() * EARTH_R_KM)
if SEPARATION == 'circle':
    assert closest > MIN_SEP_KM, f'two clusters of different folds are only {closest:.4f} km apart'
    print(f'closest clusters in different folds: {closest:.4f} km apart (limit {MIN_SEP_KM:.4f} km): OK')
else:                                     # every pair of different folds within the box diagonal: no overlap
    kidx = np.flatnonzero(kept)
    ind = BallTree(X[kidx], metric='haversine').query_radius(X[kidx], r=EPS)
    pi, pj = np.repeat(kidx, [len(v) for v in ind]), kidx[np.concatenate(ind)]
    pi, pj = pi[fold[pi] != fold[pj]], pj[fold[pi] != fold[pj]]
    assert not patches_overlap(pi, pj).any(), 'the patches of two clusters in different folds overlap'
    dx = EARTH_R_KM * np.radians(lon[pj] - lon[pi]) * np.cos(np.radians((lat[pi] + lat[pj]) / 2))
    gap = np.maximum(np.abs(dx), np.abs(EARTH_R_KM * np.radians(lat[pj] - lat[pi])))
    print(f'closest clusters in different folds: {closest:.4f} km apart; closest patches: '
          f'{gap.min() if len(gap) else np.inf:.4f} km apart in x or y (patch side + margin {BOX_KM:.2f} km): OK')

# 2. folds per country
table = fold_table(fold[kept], country_code[kept])
share = table.div(table.sum(axis=1), axis=0)
by_country = table.copy()
by_country.insert(0, 'clusters', np.bincount(country_code, minlength=len(COUNTRIES)))
by_country['removed'] = np.bincount(country_code[~kept], minlength=len(COUNTRIES))
by_country['removed_%'] = (100 * by_country['removed'] / by_country['clusters']).round(2)
by_country['min_share_%'] = (100 * share.min(axis=1)).round(1)
by_country['max_share_%'] = (100 * share.max(axis=1)).round(1)
by_country['in_all_folds'] = (table > 0).all(axis=1)
by_country['balanced'] = [outside_limits(row) == 0 for _, row in table.iterrows()]
by_country.index.name = 'country'
by_country = by_country.sort_values('clusters', ascending=False)

print(f"countries in all {N_FOLDS} folds: {by_country['in_all_folds'].sum()} of {len(by_country)} | balanced "
      f"({100 * SHARE_LO:.0f}-{100 * SHARE_HI:.0f} % per fold): {by_country['balanced'].sum()}")
for c, r in by_country[~by_country['in_all_folds']].iterrows():
    print(f'   not in all folds: {c} ({r.clusters} clusters)')
for c, r in by_country[~by_country['balanced']].iterrows():
    print(f'   not balanced: {c} ({r.clusters} clusters, fold shares {r["min_share_%"]}-{r["max_share_%"]} %)')
print(f"removed: {by_country['removed'].sum()} clusters ({100 * by_country['removed'].sum() / N:.2f} %)\n")
by_country

In [ ]:
# 3. IWI parts per fold
if STRATA:
    stable = stratum_table(fold[kept], stratum[kept])
    s_share = stable.div(stable.sum(axis=1), axis=0)
    by_stratum = stable.copy()
    by_stratum.insert(0, 'country', np.array(COUNTRIES)[STRATUM_COUNTRY])
    by_stratum.insert(1, 'clusters', stable.sum(axis=1))
    by_stratum['folds'] = (stable > 0).sum(axis=1)
    by_stratum['min_share_%'] = (100 * s_share.min(axis=1)).round(1)
    by_stratum['max_share_%'] = (100 * s_share.max(axis=1)).round(1)
    by_stratum['ok'] = [stratum_over(row) == 0 for row in stable.to_numpy()]
    # share of the part in the training folds of every test fold that holds some of it (validation fold = the one
    # before the test fold, as in 04-13); the smallest of these must be > 0, otherwise a test fold is out of reach
    val = lambda f: (f - 1) % N_FOLDS
    by_stratum['min_train_share_%'] = [round(100 * min(row[[t for t in range(N_FOLDS) if t not in (f, val(f))]].sum()
                                                       / row.sum() for f in range(N_FOLDS) if row[f] > 0), 1)
                                       for row in stable.to_numpy()]
    by_stratum.index.name = 'part'
    print(f"{STRATIFY} parts with at most {100 * STRATUM_MAX:.0f} % in one fold: {by_stratum['ok'].sum()} of "
          f"{len(by_stratum)} | in all {N_FOLDS} folds: {(by_stratum['folds'] == N_FOLDS).sum()} | in at least 3 "
          f"folds: {(by_stratum['folds'] >= 3).sum()}")
    for s, r in by_stratum[~by_stratum['ok']].iterrows():
        print(f'   crowded: {s} ({r.clusters} clusters, largest fold share {r["max_share_%"]} %)')
    worst = by_stratum['min_train_share_%'].idxmin()
    print(f"in the training folds of every test fold: at least {by_stratum['min_train_share_%'].min()} % of every "
          f"part (lowest: {worst})")
else:
    by_stratum = pd.DataFrame()
    print('STRATIFY = None: no IWI parts')
by_stratum.sort_values('max_share_%', ascending=False).head(15) if len(by_stratum) else by_stratum

In [ ]:
# 4. labels and survey years per fold
agg = {'clusters': ('cluster_id', 'size'), 'countries': ('country', 'nunique'), 'groups': ('group', 'nunique'),
       'urban_share': ('urban', 'mean')}
if 'iwi' in folds:
    agg.update(iwi_mean=('iwi', 'mean'), iwi_std=('iwi', 'std'), iwi_p10=('iwi', lambda v: v.quantile(0.1)),
               iwi_p25=('iwi', lambda v: v.quantile(0.25)), iwi_median=('iwi', 'median'),
               iwi_p75=('iwi', lambda v: v.quantile(0.75)), iwi_p90=('iwi', lambda v: v.quantile(0.9)))
if 'year' in folds:
    agg.update(year_mean=('year', 'mean'), first_year=('year', 'min'), last_year=('year', 'max'))
folds.assign(urban=URBAN[kept]).groupby('fold').agg(**agg).round(3)

In [ ]:
# 5. IWI distribution per fold: all clusters, and the countries whose fold means differ most
if 'iwi' in folds:
    fold_mean = folds.pivot_table(index='country', columns='fold', values='iwi', aggfunc='mean')
    spread = (fold_mean.max(axis=1) - fold_mean.min(axis=1)).sort_values(ascending=False)
    print('countries whose folds differ most in mean IWI (largest minus smallest fold mean):')
    print(pd.concat([fold_mean.loc[spread.index[:10]].round(1), spread.head(10).round(1).rename('range')],
                    axis=1).to_string(), '\n')
    FOLD_LINE = ['#2a78d6', '#1baf7a', '#eda100', '#008300', '#e34948']   # A-E as in the maps of section 7
    panels = [('all clusters', folds)] + [(c, folds[folds['country'] == c]) for c in spread.index[:3]]
    fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 3.6), constrained_layout=True, squeeze=False)
    for ax, (title, d) in zip(axes[0], panels):
        for f in range(N_FOLDS):
            v = np.sort(d.loc[d['fold'] == FOLDS[f], 'iwi'].to_numpy())
            if len(v):
                ax.step(v, np.arange(1, len(v) + 1) / len(v), where='post', color=FOLD_LINE[f], linewidth=1.5,
                        label=f'{FOLDS[f]}  n={len(v)}')
        ax.set_title(title, loc='left', fontsize=10)
        ax.set_xlabel('IWI')
        ax.grid(color='#e1e0d9', linewidth=0.8)
        ax.set_axisbelow(True)
        for spine in ('top', 'right'):
            ax.spines[spine].set_visible(False)
        ax.legend(frameon=False, fontsize=7, loc='lower right')
    axes[0, 0].set_ylabel('share of the fold\'s clusters with IWI below')
    plt.show()

# 6. Save

In [ ]:
# keep the earlier folds when they differ: every run made with them has to be redone in a new run folder
if os.path.exists(OUT_CSV):
    old = pd.read_csv(OUT_CSV, usecols=['cluster_id', 'fold'])
    same = (old['cluster_id'].astype(str).tolist() == folds['cluster_id'].astype(str).tolist()
            and old['fold'].astype(str).tolist() == folds['fold'].astype(str).tolist())
    if not same:
        stamp = time.strftime('%Y%m%d_%H%M', time.localtime(os.path.getmtime(OUT_CSV)))
        backup = OUT_CSV.replace('.csv', f'_until_{stamp}.csv')
        os.replace(OUT_CSV, backup)
        print(f'The folds changed. The earlier ones are now in {backup}.\n'
              'Move the run folders of 04 to 13 aside before training again: the notebooks skip every run '
              'whose result.json exists, even one made with the earlier folds.\n')
folds.to_csv(OUT_CSV, index=False)
removed.to_csv(REMOVED_CSV, index=False)
by_country.to_csv(COUNTRY_CSV)
by_stratum.to_csv(STRATUM_CSV)
for path in (OUT_CSV, REMOVED_CSV, COUNTRY_CSV, STRATUM_CSV):
    print(path)

# 7. Maps

One map per fold (coloured) over all clusters (grey), and the removed clusters. Then the countries with the most removed clusters, with all five folds together and a 9.50 km bar. Each fold also has its own marker shape, so the folds can be told apart without colour. The last cell is an interactive folium map of one country, like the one in `ToBe_07`, with a distance tool (`MAP_COUNTRY`).

In [ ]:
FOLD_COLORS  = ['#2a78d6', '#1baf7a', '#eda100', '#008300', '#e34948']   # A-E: blue, aqua, yellow, green, red
FOLD_MARKERS = ['o', 's', '^', 'D', 'v']
INK, MUTED, CONTEXT = '#0b0b0b', '#52514e', '#d9d8d2'


def style(ax, aspect_lat):
    ax.set_aspect(1 / np.cos(np.radians(aspect_lat)))      # degrees of longitude are shorter than of latitude
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_color(CONTEXT)


fig, axes = plt.subplots(2, 3, figsize=(15, 10), constrained_layout=True)
for f, ax in enumerate(axes.flat):
    ax.scatter(lon[kept], lat[kept], s=1, c=CONTEXT, lw=0, rasterized=True)
    if f < N_FOLDS:
        m = kept & (fold == f)
        ax.scatter(lon[m], lat[m], s=3, c=FOLD_COLORS[f], lw=0, rasterized=True)
        ax.set_title(f"Fold {FOLDS[f]}  ·  {m.sum():,} clusters, {df.loc[m, 'country'].nunique()} countries",
                     color=INK, fontsize=11, loc='left')
    else:
        ax.scatter(lon[~kept], lat[~kept], s=10, c=INK, marker='x', lw=0.8)
        ax.set_title(f'Removed  ·  {(~kept).sum():,} clusters', color=INK, fontsize=11, loc='left')
    style(ax, np.median(lat))
fig.savefig(MAP_PNG, dpi=150)
plt.show()
print(MAP_PNG)

In [ ]:
def country_map(ax, name):
    m = (df['country'] == name).to_numpy()
    for f in range(N_FOLDS):
        s = m & kept & (fold == f)
        ax.scatter(lon[s], lat[s], s=16, c=FOLD_COLORS[f], marker=FOLD_MARKERS[f], lw=0,
                   label=f'{FOLDS[f]}  {s.sum()}')
    r = m & ~kept
    ax.scatter(lon[r], lat[r], s=16, c=INK, marker='x', lw=0.8, label=f'removed  {r.sum()}')
    # an empty strip below the clusters for the MIN_SEP_KM scale bar
    y0, y1 = lat[m].min(), lat[m].max()
    pad = 0.04 * (y1 - y0) + 0.02
    ax.set_ylim(y0 - 4 * pad, y1 + pad)
    ax.set_xlim(lon[m].min() - pad, lon[m].max() + pad)
    style(ax, lat[m].mean())
    x0 = lon[m].min()
    bar = MIN_SEP_KM / (111.32 * np.cos(np.radians(lat[m].mean())))    # 9.50 km in degrees of longitude
    ax.plot([x0, x0 + bar], [y0 - 2.5 * pad] * 2, color=INK, lw=2, solid_capstyle='butt')
    ax.text(x0 + bar + pad / 2, y0 - 2.5 * pad, f'{MIN_SEP_KM:.1f} km', color=MUTED, va='center', fontsize=9)
    ax.set_title(f'{name}  ·  {m.sum():,} clusters', color=INK, fontsize=11, loc='left')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 0), ncol=3, fontsize=8, frameon=False,
              labelcolor=MUTED, markerscale=1.2)


show = by_country.sort_values('removed', ascending=False).index[:4]
fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 6), constrained_layout=True, squeeze=False)
for ax, name in zip(axes[0], show):
    country_map(ax, name)
plt.show()

In [ ]:
from IPython.display import display

if folium is None:
    print('folium is not installed: no interactive map')
else:
    name = MAP_COUNTRY or by_country['removed'].idxmax()
    m = np.flatnonzero((df['country'] == name).to_numpy())
    fmap = folium.Map(location=[lat[m].mean(), lon[m].mean()], zoom_start=7)
    for i in m:
        label = FOLDS[fold[i]] if kept[i] else 'removed'
        color = FOLD_COLORS[fold[i]] if kept[i] else INK
        folium.CircleMarker([lat[i], lon[i]], radius=4, color=color, weight=1, fill=True, fill_opacity=0.8,
                            tooltip=f"{df.at[i, 'cluster_id']} | fold {label}").add_to(fmap)
    fmap.add_child(MeasureControl())
    display(fmap)